In [1]:
import torch


In [2]:
import pandas as pd
import numpy as np
import os
import hashlib
import requests

In [ ]:
DATA_HUB=dict()
DATA_URL='http://d2l-data.s3-accelerate.amazonaws.com/'

def download(name,cache_dir=os.path.join('..','data')):
    assert name in DATA_HUB,f"{name}不存在于{DATA_HUB}"
    url,sha1_hash=DATA_HUB[name]
    os.makedirs(cache_dir,exist_ok=True)
    fname=os.path.join(cache_dir,url.split('/')[-1])
    if os.path.exists(fname):
        shal=hashlib.sha1()
        with open(fname,'rb') as f:
            while True:
                data=f.read(1048576)
                if not data:
                    break
                shal.update(data)
            if shal.hexdigest()==sha1_hash:
                return fname
    print(f'正在下载{url}到{fname}')
    r=requests.get(url,stream=True,verify=True)
    with open(fname,'wb') as f:
        f.write(r.content)
    return fname

device(type='cpu')

In [ ]:
DATA_HUB['kaggle_house_train'] = (  #@save
    DATA_URL + 'kaggle_house_pred_train.csv',
    '585e9cc93e70b39160e7921475f9bcd7d31219ce')

DATA_HUB['kaggle_house_test'] = (  #@save
    DATA_URL + 'kaggle_house_pred_test.csv',
    'fa19780a7b011d9b009e8bff8e99922a8ee2eb90')

In [ ]:
train_data = pd.read_csv(download("kaggle_house_train"))
test_data = pd.read_csv(download("kaggle_house_test"))

print(train_data.shape)
print(test_data.shape)

print(train_data.iloc[0:4,[0,1,2,3,-3,-2,-1]])

In [ ]:
all_features = pd.concat((train_data.iloc[:,1:], test_data.iloc[:,1:]))

In [ ]:
numeric_features = all_features.dtypes[all_features.dtypes != 'object'].index

In [ ]:
# 第1步：先填充缺失值
all_features[numeric_features] = all_features[numeric_features].fillna(
    all_features[numeric_features].mean()
)

# 第2步：再标准化
all_features[numeric_features] = all_features[numeric_features].apply(
    lambda x: (x - x.mean()) / x.std(), axis=0
)

In [ ]:
all_features = pd.get_dummies(all_features, dummy_na=True,dtype=np.float32)
all_features.shape

In [ ]:
train_features=torch.tensor(all_features[:train_data.shape[0]].to_numpy(dtype=np.float32),dtype=torch.float32)
test_features=torch.tensor(all_features[train_data.shape[0]:].to_numpy(dtype=np.float32),dtype=torch.float32)
train_labels=torch.tensor(train_data.SalePrice.values.reshape(-1,1),dtype=torch.float32)

In [ ]:
from d2l import torch as d2l

In [ ]:
loss=torch.nn.MSELoss()

net=torch.nn.Sequential(torch.nn.Linear(train_features.shape[1],1))

optimizer=torch.optim.Adam(net.parameters(),lr=5,weight_decay=0)

def log_rmse(net,features,labels):
    clipped=torch.clamp(net(features),1,float('inf'))
    return torch.sqrt(loss(torch.log(clipped),torch.log(labels)))

def train(net,loss,optimizer,train_features,train_labels,test_features,test_labels,minibatch_size=20,epochs=100):
    train_iter=torch.utils.data.DataLoader(torch.utils.data.TensorDataset(train_features,train_labels),batch_size=minibatch_size,shuffle=True)
    for epoch in range(epochs):
        for X,y in train_iter:
            l=loss(net(X),y)
            optimizer.zero_grad()
            l.backward()
            optimizer.step()
        rmse=log_rmse(net,train_features,train_labels)
        print(f'epoch{epoch+1},test rmse{rmse:f}')

In [ ]:
def get_k_fold_data(k, i, X, y):
    assert k > 1
    fold_size = X.shape[0] // k
    X_train, y_train = None, None
    for j in range(k):
        idx = slice(j * fold_size, (j + 1) * fold_size)
        X_part, y_part = X[idx, :], y[idx]
        if j == i:
            X_valid, y_valid = X_part, y_part
        elif X_train is None:
            X_train, y_train = X_part, y_part
        else:
            X_train = torch.cat([X_train, X_part], 0)
            y_train = torch.cat([y_train, y_part], 0)
    return X_train, y_train, X_valid, y_valid


In [ ]:
def k_fold(net,loss,optimizer,k, X_train, y_train, num_epochs,batch_size):
    for i in range(k):
        data = get_k_fold_data(k, i, X_train, y_train)
        train(net, loss, optimizer, data[0], data[1],data[2],data[3],batch_size, num_epochs)

In [ ]:
k, num_epochs, lr, weight_decay, batch_size = 5, 100, 5, 0, 64
k_fold(net,loss,optimizer,k, train_features, train_labels, num_epochs, batch_size)

In [ ]:
preds = net(test_features).detach().numpy()
# 将其重新格式化以导出到Kaggle
test_data['SalePrice'] = pd.Series(preds.reshape(1, -1)[0])
submission = pd.concat([test_data['Id'], test_data['SalePrice']], axis=1)
submission.to_csv('submission.csv', index=False)